In [1]:
import numpy as np
import pandas as pd
import glob, os, subprocess, vcf, shutil, sparse, yaml, sys, pickle
from Bio import Entrez, Seq, SeqIO
import scipy.stats as st

# utils files are in a separate folder
sys.path.append("utils")
from saliency_utils import *
from inSilicoMut_utils import *
from analysis_utils import *

from sklearn.linear_model import Ridge, RidgeCV

BASE_TO_COLUMN = {'A': 0, 'C': 1, 'T': 2, 'G': 3, '-': 4}
data_dir = "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs"
h37Rv_genes = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/H37Rv/mycobrowser_h37rv_genes_v4.csv")

4411532


In [105]:
MXF_Reg_stats = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/CNN_results/MXF_gyrBA/validation/LinReg_stats.csv")
MXF_CNN_stats = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/CNN_results/MXF_gyrBA/validation/CNN_stats.csv")

ridge_results = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/CNN_results/MXF_gyrBA/ridge/ridge_results.csv")

In [101]:
ridge_results.query("CV==0")

,Drug,Model,Num_Loci,Binned_MAE,Binned_MSE,MAE,MSE,Within_1Bin,Within_doubling,Spearman,Pearson,Sensitivity,Specificity,Precision,Accuracy,Balanced_Acc,CV,Lineage
0,MXF,LinReg,1,0.502056,0.842966,0.950762,1.736435,0.807428,0.838377,0.558994,0.758399,0.860169,0.975369,0.871245,0.956671,0.917769,0,0


In [103]:
# MXF_isolate_variants = pd.read_csv(os.path.join(data_dir, "MXF", "isolate_variants.csv.gz"), compression="gzip")
# len(MXF_isolate_variants.Isolate.unique())

In [15]:
df_val = pd.read_csv(os.path.join(data_dir, "MXF", "validation_data.csv"))

In [14]:
MXF_isolate_variants.query("Isolate in @df_val.ROLLINGDB_ID")

,POS,REF,ALT,FILTER,AF,GENE,EFFECT,NUC,PROT,Isolate


In [79]:
isolate_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/trainVal_isolateVariants_AllDrugs.tsv", usecols=["Isolate"], sep="\t")
len(isolate_variants.Isolate.unique())

12210

In [80]:
11115+1769-12210

674

In [3]:
all_isolates = os.listdir("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF")

In [20]:
len(set(all_isolates) - set(isolate_variants.Isolate))

1769

In [38]:
rollingDB_isolates = os.listdir('/n/data1/hms/dbmi/farhat/rollingDB/genomic_data')
cryptic_isolates = os.listdir('/n/data1/hms/dbmi/farhat/rollingDB/cryptic_output')

cryptic_to_clean = list(set(all_isolates).intersection(cryptic_isolates))
rollingDB_to_clean = list(set(all_isolates).intersection(rollingDB_isolates))
len(cryptic_to_clean), len(rollingDB_to_clean)

(9340, 1786)

In [73]:
def move_vcf_files(sample_id, output_dir):

    remove_files = glob.glob(f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/pilon/*")
    
    if not os.path.isfile(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{sample_id}/pilon/{sample_id}.vcf"):
        print(f"Already cleaned vcf files for {sample_id}")
        return None

    else:
        if os.path.isfile(f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/pilon/{sample_id}.vcf"):
            os.remove(f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/pilon/{sample_id}.vcf")
        
            shutil.move(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{sample_id}/pilon/{sample_id}.vcf", f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/pilon/{sample_id}.vcf")

    # shutil.move(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{sample_id}/pilon", f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}")

In [75]:
# for i, sample in enumerate(cryptic_to_clean):
#     move_vcf_files(sample, "cryptic_output")
#     if i % 1000 == 0:
#         print(i)

Already cleaned vcf files for ERR4812043
0
1000
2000
3000
4000
5000
6000
7000
8000
9000


In [76]:
# for i, sample in enumerate(rollingDB_to_clean):
#     move_vcf_files(sample, "genomic_data")
#     if i % 1000 == 0:
#         print(i)

0
1000


In [78]:
for isolate in all_isolates:
    if not os.path.isfile(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{isolate}/pilon/{isolate}.eff.vcf"):
        print(isolate)

In [71]:
f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{sample}/pilon/{sample}_full.vcf.gz"

'/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/ERR4810743/pilon/ERR4810743_full.vcf.gz'

In [53]:
def move_bam_files(sample_id, output_dir):

    remove_files = glob.glob(f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/bam/*")

    if f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/bam/{sample_id}.dedup.bam.metrics" in remove_files:
        print(f"Already cleaned bam files for {sample_id}")
        return None

    # remove the entire bam folder, then move the new one to its place
    if os.path.isdir(f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/bam"):
        shutil.rmtree(f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}/bam")
    
    shutil.move(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{sample_id}/bam", f"/n/data1/hms/dbmi/farhat/rollingDB/{output_dir}/{sample_id}")

In [34]:
# for i, sample_id in enumerate(cryptic_to_clean):
    
#     move_bam_files(sample_id)

#     if i % 1000 == 0:
#         print(i)

Already cleaned bam files for ERR4812043
0
Already cleaned bam files for ERR4831464
1000
2000
3000
4000
5000
6000
7000
8000
9000


In [54]:
# for i, sample_id in enumerate(rollingDB_to_clean):

#     if 'bam' in os.listdir(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{sample_id}"):
#         move_bam_files(sample_id, "genomic_data")

#     if i % 100 == 0:
#         print(i)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700


In [8]:
list(set(all_isolates).intersection(cryptic_isolates))[:10]

['ERR4812043',
 'ERR4831464',
 'ERR4812471',
 'ERR4810959',
 'ERR4813341',
 'ERR4811624',
 'ERR4830499',
 'ERR2184409',
 'ERR4797197',
 'ERR4830447']

In [40]:
os.listdir("/n/data1/hms/dbmi/farhat/rollingDB/genomic_data/SAMEA3558221/bam")

['SAMEA3558221.sorted.duprem.bam', 'SAMEA3558221.sorted.duprem.bam.bai']

False

In [22]:
pd.Series(list(set(all_isolates) - set(isolate_variants.Isolate))).to_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/fNames.txt", index=False, header=None, sep="\t")

In [23]:
missing_isolates = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/fNames.txt", sep="\t", header=None)[0].values
len(missing_isolates)

1769

In [79]:
lineages = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/lineage_matrix_Coll2014.csv", index_col=[0])
lineages.shape, len(lineages.index.unique())

((12884, 62), 12884)

In [80]:
# new_names = []

# for name in lineages.index.values:
#     new_names.append(name.replace("_", "-"))

# assert len(new_names) == len(lineages)
# lineages.index = new_names

In [81]:
set(df_val.ROLLINGDB_ID) - set(lineages.index)

set()

In [69]:
df_val

,ROLLINGDB_ID,ISOLATION_LOCATION,DB_OF_ORIGIN,STUDY_NAME,STUDY_PMID,TESTING_LOCATION,MEDIA,MXF_quality,MXF_lower_bound,MXF_midpoint,MXF_upper_bound
0,JPN-R2012-0001,Japan,JATA-KIT,NaN,NaN,Not specified,MGIT960/MGIT7h9,NaN,8.000,12.0000,16.000
1,JPN-R2012-0002,Japan,JATA-KIT,NaN,NaN,Not specified,MGIT960/MGIT7h9,NaN,1.000,1.5000,2.000
2,JPN-R2012-0003,Japan,JATA-KIT,NaN,NaN,Not specified,MGIT960/MGIT7h9,NaN,0.500,0.7500,1.000
3,JPN-R2012-0004,Japan,JATA-KIT,NaN,NaN,Not specified,MGIT960/MGIT7h9,NaN,2.000,3.0000,4.000
4,JPN-R2012-0005,Japan,JATA-KIT,NaN,NaN,Not specified,MGIT960/MGIT7h9,NaN,0.120,0.1800,0.240
...,...,...,...,...,...,...,...,...,...,...,...
1329,C478,NaN,CASS,NaN,NaN,Not specified,MYCOTB,NaN,0.030,0.0460,0.062
1330,C480,NaN,CASS,NaN,NaN,Not specified,MYCOTB,NaN,0.500,0.7500,1.000
1331,C482,NaN,CASS,NaN,NaN,Not specified,MYCOTB,NaN,0.008,0.0115,0.015
1332,C485,NaN,CASS,NaN,NaN,Not specified,MYCOTB,NaN,0.094,0.1095,0.125


In [32]:
# YES, ALL THE MIC-ML VCFs ARE GOOD

# for isolate in missing_isolates:

#     fName = f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF/{isolate}/pilon/{isolate}.eff.vcf"
    
#     if not os.path.isfile(fName):
#         print(isolate)
#     else:
#         line_count = subprocess.Popen(f'grep -v "^##" {fName} | wc -l', shell=True, encoding='utf8', stdout=subprocess.PIPE)
#         line_count = line_count.communicate()[0]
#         line_count = int(line_count.replace("\n", ""))
        
#         if line_count == 0:
#             print(isolate)

In [ ]:
def mutation_catalog_with_bootstrapping(df, drug, who_variants_df, isolate_variants_df, binary_thresh, return_stats=["Sensitivity", "Specificity", "AUC", "Accuracy", "Balanced_Acc"]):
    
    df = df.rename(columns={"ROLLINGDB_ID": "Isolate"}).reset_index(drop=True)
    cat1_mutations = who_variants_df.query("drug == @drug & confidence=='1) Assoc w R'").mutation.values
    isolates_R = isolate_variants_df.query("mutation in @cat1_mutations & FILTER == 'PASS' & Isolate in @df.Isolate.values").Isolate.values
        
    df_pred_catalog = df[["Isolate", f"{drug}_midpoint"]]
    df_pred_catalog["y_test"] = (df[f"{drug}_midpoint"] > binary_thresh).astype(int)
    df_pred_catalog["y_pred"] = df_pred_catalog["Isolate"].map(dict(zip(isolates_R, np.ones(len(isolates_R))))).fillna(0).astype(int)
    
    df_stats = compute_binary_metrics(df_pred_catalog["y_test"], df_pred_catalog["y_pred"], binary_thresh, binarize=False)[return_stats]
    df_stats["CV"] = 0
    bs_lst = []
    
    # perform bootstrapping with 10 replicates
    for i in range(10):
        
        bs_sample_idx = np.random.choice(df.index.values, size=len(df), replace=True)
        bs_df = df.iloc[bs_sample_idx, :]
        bs_isolates_R = isolate_variants_df.query("mutation in @cat1_mutations & FILTER == 'PASS' & Isolate in @bs_df.Isolate.values").Isolate.values
        
        bs_pred_catalog = bs_df[["Isolate", f"{drug}_midpoint"]]
        bs_pred_catalog["y_test"] = (bs_df[f"{drug}_midpoint"] > binary_thresh).astype(int)
        bs_pred_catalog["y_pred"] = bs_pred_catalog["Isolate"].map(dict(zip(bs_isolates_R, np.ones(len(bs_isolates_R))))).fillna(0).astype(int)
        
        bs_df_stats = compute_binary_metrics(bs_pred_catalog["y_test"], bs_pred_catalog["y_pred"], binary_thresh, binarize=False)[return_stats]
        bs_df_stats["CV"] = i + 1
        bs_lst.append(bs_df_stats)

    df_return = pd.concat([df_stats, pd.concat(bs_lst, axis=0)], axis=0).reset_index(drop=True)
    df_return["Model"] = "Catalog"
    return df_return

In [57]:
megapipe_samples.loc[megapipe_samples[0].isin(need_to_fix)]

,0,1,2
2113,ERR4812738,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
4225,ERR4829184,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
6365,ERR4811895,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
8039,ERR4831349,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
8467,ERR4810716,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...


In [90]:
train_isolate_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/train_isolate_variants.tsv", sep="\t")
len(train_isolate_variants.Isolate.unique())

11115

In [92]:
train_isolate_variants.loc[pd.isnull(train_isolate_variants["Isolate"])]

,POS,REF,ALT,FILTER,AF,GENE,EFFECT,NUC,PROT,Isolate


In [93]:
train_isolate_variants.query("Isolate=='ERR4831349'")

,POS,REF,ALT,FILTER,AF,GENE,EFFECT,NUC,PROT,Isolate
20204568,1977,A,G,PASS,1.0,dnaA-dnaN,intergenic_region,n.1977A>G,.,ERR4831349
20204569,3801,G,C,PASS,1.0,recF,synonymous_variant,c.522G>C,p.Arg174Arg,ERR4831349
20204570,4013,T,C,PASS,1.0,recF,missense_variant,c.734T>C,p.Ile245Thr,ERR4831349
20204571,7362,G,C,PASS,1.0,gyrA,missense_variant,c.61G>C,p.Glu21Gln,ERR4831349
20204572,7585,G,C,PASS,1.0,gyrA,missense_variant,c.284G>C,p.Ser95Thr,ERR4831349
...,...,...,...,...,...,...,...,...,...,...
20206105,4399185,C,T,PASS,1.0,Rv3910,synonymous_variant,c.2589C>T,p.Arg863Arg,ERR4831349
20206106,4400246,G,A,PASS,1.0,sigM,missense_variant,c.61G>A,p.Asp21Asn,ERR4831349
20206107,4400660,AC,A,PASS,0.99,sigM,frameshift_variant,c.478delC,p.Arg160fs,ERR4831349
20206108,4404034,C,G,PASS,1.0,cwlM,synonymous_variant,c.843C>G,p.Gly281Gly,ERR4831349


In [79]:
train_isolate_variants.query("Isolate=='ERR5917707'").FILTER.value_counts()

FILTER
PASS              1753
Amb                171
LowCov              99
Amb;LowCov          72
Del;LowCov          37
Del                 19
Del;Amb;LowCov      16
Del;Amb             13
Name: count, dtype: int64

In [81]:
train_isolate_variants.query("Isolate=='ERR5917707' & FILTER.str.contains('LowCov')").GENE.unique()

array(['pstP', 'Rv0095c', 'nrdB-gabD1', 'Rv0278c', 'Rv0278c-PE_PGRS4',
       'PE_PGRS4', 'Rv0493c', 'PE_PGRS6', 'PE_PGRS9', 'PE_PGRS10',
       'Rv0823c', 'PE_PGRS13', 'PE_PGRS15', 'PE_PGRS16', 'PE_PGRS20',
       'PE_PGRS21', 'PE_PGRS22', 'Rv1148c', 'PE_PGRS23', 'PE_PGRS24',
       'PE_PGRS26', 'PE_PGRS27', 'PE_PGRS28', 'PPE24', 'wag22', 'Rv1765c',
       'Rv1765c-Rv1766', 'PE_PGRS33', 'PPE34', 'Rv1929c', 'Rv2015c',
       'Rv2082', 'PE_PGRS38', 'Rv2248-glpD1', 'PPE39', 'PE_PGRS43',
       'Rv2507', 'lppA', 'lppB', 'PE_PGRS45', 'PE_PGRS47',
       'Rv2813-Rv2816c', 'PE_PGRS48', 'esxQ', 'Rv3191c-metU', 'PPE54',
       'PE_PGRS50', 'Rv3359', 'PE_PGRS52', 'PE_PGRS53', 'PE_PGRS54',
       'PE_PGRS55', 'PE_PGRS55-PE_PGRS56', 'PE_PGRS56', 'PE_PGRS57',
       'hsaD', 'PE_PGRS58', 'PE_PGRS61'], dtype=object)

In [61]:
train_isolate_variants.query("Isolate in ['ERR4831349']").FILTER.value_counts()

FILTER
LowCov    556970
Name: count, dtype: int64

In [63]:
train_isolate_variants.query("Isolate in ['ERR5917707']").FILTER.value_counts()

FILTER
PASS          263255
LowCov          4367
Del              246
Del;LowCov       209
Del;Amb            2
Amb                2
Name: count, dtype: int64

In [64]:
train_isolate_variants.shape, train_isolate_variants.query("Isolate not in ['ERR4831349', 'ERR5917707']").shape

((21027439, 10), (20202388, 10))

In [78]:
for drug in ["RIF", "INH", "MXF", "EMB"]:

    # for fName in glob.glob(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/fastas/*.fasta"):
    #     os.remove(fName)

    # shutil.rmtree(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/VCF_QC_files")

In [66]:
# train_isolate_variants.query("Isolate not in ['ERR4831349', 'ERR5917707']").to_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/train_isolate_variants.tsv", sep="\t", index=False)

In [87]:
pd.Series(['ERR4831349']).to_csv('/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/fNames.txt', sep='\t', index=False, header=None)

In [35]:
megapipe_samples = pd.read_csv("megapipe_samples.tsv", sep="\t", header=None)

In [58]:
megapipe_samples.loc[megapipe_samples[0]=="ERR5917707"].to_csv("megapipe_test.tsv", sep="\t", index=False, header=None)

In [30]:
for isolate in temp_seq.keys():
    if len(''.join(temp_seq[isolate])) > 5000:
        print(isolate)

ERR5917707


In [18]:
len(set(mxf_isolates) - set(lev_isolates))

2442

In [12]:
insertion_sites.len_insertion.sum()

14739

In [9]:
insertion_sites

,Unnamed: 0,idx,len_insertion
0,0,0,3
1,1,1,3
2,2,2,3
3,3,3,3
4,4,4,3
...,...,...,...
4908,4908,4911,3
4909,4909,4912,3
4910,4910,4913,3
4911,4911,4914,3


In [45]:
kwargs = yaml.safe_load((open("config_files/config_pza.yaml")))

drug = kwargs["drug"]
include_lineage = kwargs["include_lineage"]
binary_thresh = kwargs["binary_thresh"]
locus_list = kwargs["locus_list"]
num_loci = len(locus_list)
loss_type = kwargs["loss_type"]
fasta_dir = kwargs["genotype_input_directory"]

df_phenos = pd.read_csv(kwargs['phenotype_file'])

out_dir = kwargs["output_path"]
ridge_dir = os.path.join(out_dir, "ridge")
bootstrap_dir = os.path.join(out_dir, "ridge", "bootstrapping")
    
if not os.path.isdir(bootstrap_dir):
    os.makedirs(bootstrap_dir)

gene_coords, _ = get_gene_coords(locus_list, fasta_dir)
h37Rv_coords = make_h37rv_coordinates(gene_coords, locus_list, fasta_dir)

# read in matrices of input sequences
train_matrix = sparse.load_npz(f"{out_dir}/pkl_sparse_train.npz").todense()
test_matrix = sparse.load_npz(f"{out_dir}/pkl_sparse_test.npz").todense()
ref_matrix = sparse.load_npz(f"{out_dir}/pkl_sparse_ref.npz").todense()

# make dataframes of coordinates
gene_coords, _ = get_gene_coords(locus_list, fasta_dir)
h37Rv_coords = make_h37rv_coordinates(gene_coords, locus_list, fasta_dir)

In [65]:
def get_single_locus_Reg_input(locus, locus_list, train_matrix, test_matrix, ref_matrix):

    locus_idx = locus_list.index(locus)

    train_samples = train_matrix.shape[0]
    test_samples = test_matrix.shape[0]
    
    one_hot_encodings = train_matrix.shape[1]
    longest_locus = train_matrix.shape[2]
    num_loci = train_matrix.shape[3]
    assert one_hot_encodings == 5

    # turn the matrices into dataframes for easy manipulation
    df_train = pd.DataFrame(np.reshape(train_matrix[:, :, :, locus_idx], (train_samples, one_hot_encodings * longest_locus), order='F'))
    df_test = pd.DataFrame(np.reshape(test_matrix[:, :, :, locus_idx], (test_samples, one_hot_encodings * longest_locus), order='F'))
    df_ref = pd.DataFrame(np.reshape(ref_matrix[:, :, :, locus_idx], (1, one_hot_encodings * longest_locus), order='F'))

    # need to get all the nucleotide positions to name the columns. This makes manipulation easier and is also useful to keep track of which positions went into the model (interpretability)
    # k is an iterator to keep track of indels
    seq_coords = []
    k = 0
    
    for coord in h37Rv_coords[:, locus_idx]:

        # indels -- position is NaN, so give unique names that are a concatenation of the locus and an index
        if pd.isnull(coord):
            coord = f"{locus}_{k}"
            k += 1
        else:
            coord = str(int(coord))
            
        seq_coords += [f"{coord}_{nuc}" for nuc in BASE_TO_COLUMN.keys()]
    
    assert len(seq_coords) == len(df_train.columns)
    df_train.columns = seq_coords
    df_test.columns = seq_coords

    # this is a dataframe of length 1
    df_ref.columns = seq_coords
    h37Rv_ref_seq = df_ref[df_ref.columns[(df_ref.loc[0] == 1)]]

    # keep only variables that are not the same everywhere because there is no signal
    df_train_keep = df_train.loc[:, df_train.nunique() > 1]    
    keep_pos = ['_'.join(val.split("_")[:-1]) for val in df_train_keep.columns]
    
    # when value_counts = 1, it's for indels, where the only options are indel or not. The four nucleotides are 0 for all samples and get dropped in the previous step
    single_allele_pos =  pd.Series(keep_pos).value_counts()[pd.Series(keep_pos).value_counts() == 2].index.values
    multi_allele_pos = pd.Series(keep_pos).value_counts()[pd.Series(keep_pos).value_counts() > 2].index.values
    
    # for positions with only two alleles (REF and ALT, essentially), only need to keep one because they are redundant information and perfectly correlated
    single_allele_pos_keep_cols = []
    
    # preferentially keep the alternative allele because it makes interpretability easier
    for pos in single_allele_pos:

        single_pos_cols = [col for col in df_train_keep.columns if "_".join(col.split("_")[:-1]) == pos]
        df_keep_single_pos = df_train_keep[single_pos_cols]
        
        alt_col = set(df_keep_single_pos.columns) - set(h37Rv_ref_seq[h37Rv_ref_seq.columns[h37Rv_ref_seq.columns.str.contains(pos)]].columns)
        assert len(alt_col) == 1
        single_allele_pos_keep_cols += list(alt_col)
    
    assert len(single_allele_pos_keep_cols) == len(single_allele_pos)

    # create training and testing dataframes from the columns to keep (which is based on df_train only)
    df_train_final = pd.concat([df_train_keep[single_allele_pos_keep_cols], df_train_keep[df_train_keep.columns[df_train_keep.columns.str.contains('|'.join(multi_allele_pos))]]], axis=1)
    
    # for the test dataframe, keep only the columns determined from the train dataframe
    df_test_final = df_test[df_train_final.columns]

    return df_train_final, df_test_final

In [64]:
X_train = []
X_test = []

for locus in locus_list:
    single_locus_train, single_locus_test = get_single_locus_Reg_input(locus, locus_list, train_matrix, test_matrix, ref_matrix)

    X_train.append(single_locus_train)
    X_test.append(single_locus_test)

X_train = pd.concat(X_train, axis=1)
X_test = pd.concat(X_test, axis=1)

# no changes were made to sample ordering, so use the exact indexes of the isolates from df_phenos
X_train.index = df_phenos.query("category=='original_train_set'")["ROLLINGDB_ID"]
X_test.index = df_phenos.query("category=='original_test_set'")["ROLLINGDB_ID"]

X_train.shape, X_test.shape

((7304, 1151), (1827, 1151))

In [29]:
class CustomRidgeCV(RidgeCV):
                
    def fit(self, X, y, loss_type=None, lower_bounds=None, upper_bounds=None, *args, **kwargs):
        
        self.loss_type = loss_type
        self.lower_bounds = lower_bounds
        self.upper_bounds = upper_bounds
        
        super().fit(X, y, *args, **kwargs)
        
    def score(self, X, y, loss_type=None, lower_bounds=None, upper_bounds=None):
        
        self.loss_type = loss_type
        self.lower_bounds = lower_bounds
        self.upper_bounds = upper_bounds
        
        def boundedLoss_Reg_L2penalty(y_pred, y_true):

            '''
            y_test and y_pred are log2-transformed. lower_bounds and upper_bounds are NOT
            
            loss_type is L1 or L2, specifying whether to return the MAE or MSE, NOT THE TYPE OF REGULARIZATION. THIS FUNCTION ONLY USES L2 REGULARIZATION
            
            reg_param is the strength of regularization to apply -- multiply the sum of the squares of sample_weights by this term
            '''

            # get predictions, then exponentiate to get actual MICs
            y_pred_MIC = np.exp2(y_pred)

            # compute the errors first using the log-MICs, based on the desired loss type
            if self.loss_type == "L1":
                errors = np.abs(y_true - y_pred)
            elif self.loss_type == "L2":
                errors = (y_true - y_pred)**2
            else:
                raise RuntimeError(f"{self.loss_type} is not a valid loss function type")

            # compute error using only the points that are predicted outside of their bin. Sum the errors, then divide by the number of points
            binned_error = np.sum(errors[((y_pred_MIC < self.lower_bounds) | (y_pred_MIC > self.upper_bounds))]) / len(y_pred_MIC)
            return binned_error + self.alpha * np.sum(np.square(self.coef_))
        
        y_pred = self.predict(X)
        return -boundedLoss_Reg_L2penalty(y_pred, y)

    

def boundedLoss_Reg(y_pred, y_true, lower_bounds, upper_bounds, loss_type="L2"):

    '''
    y_test and y_pred are log2-transformed. lower_bounds and upper_bounds are NOT
    loss_type is L1 or L2, specifying whether to return the MAE or MSE
    reg_param is the strength of regularization to apply -- multiply the sum of the squares of sample_weights by this term
    '''

    # get predictions, then exponentiate to get actual MICs
    y_pred_MIC = np.exp2(y_pred)

    # compute the errors first using the log-MICs, based on the desired loss type
    if loss_type == "L1":
        errors = np.abs(y_true - y_pred)
    elif loss_type == "L2":
        errors = (y_true - y_pred)**2
    else:
        raise RuntimeError(f"{loss_type} is not a valid loss function type")

    # compute error using only the points that are predicted outside of their bin. Sum the errors, then divide by the number of points
    return np.sum(errors[((y_pred_MIC < lower_bounds) | (y_pred_MIC > upper_bounds))]) / len(y_pred_MIC)


    
    
class CustomRidge(Ridge):
                
    def fit(self, X, y, loss_type=None, lower_bounds=None, upper_bounds=None, *args, **kwargs):
        
        self.loss_type = loss_type
        self.lower_bounds = lower_bounds
        self.upper_bounds = upper_bounds
        
        super().fit(X, y, *args, **kwargs)

In [30]:
def ridge_mic(X_train, X_test, df_phenos, drug, include_lineage, binary_thresh, num_loci, num_bootstrap=10):

    df_train = df_phenos.query("category=='original_train_set'").reset_index(drop=True)
    df_test = df_phenos.query("category=='original_test_set'").reset_index(drop=True)
    
    if include_lineage:
        print(f"Fitting model with lineages")
        # lineages = pd.get_dummies(df_phenos["Lineage"])
        # lineages.index = df_phenos["ROLLINGDB_ID"]
        
        lineages = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/lineage_matrix_Coll2014.csv", index_col=[0])
        assert len(np.unique(lineages.values)) == 2
        lineages = lineages.loc[df_phenos["ROLLINGDB_ID"]]

        X_train = X_train.merge(lineages, left_index=True, right_index=True, how="left")
        X_test = X_test.merge(lineages, left_index=True, right_index=True, how="left")
        
        # X_train = np.concatenate([df_train.values, lineages.loc[df_train.index.values, :].values], axis=1)
        # X_test = np.concatenate([df_test.values, lineages.loc[df_test.index.values, :].values], axis=1)

    X_train = X_train.values
    X_test = X_test.values
        
    # perform mean / SD scaling using the training set
    train_mean = X_train.mean()
    train_sd = X_train.std()
        
    X_train = (X_train - train_mean) / train_sd
    X_test = (X_test - train_mean) / train_sd
    y_train = np.log2(df_train[f"{drug}_midpoint"].values)
    y_test = np.log2(df_test[f"{drug}_midpoint"].values)
    
    lower_bounds_train, upper_bounds_train = df_train[f"{drug}_lower_bound"].values, df_train[f"{drug}_upper_bound"].values
    lower_bounds_test, upper_bounds_test = df_test[f"{drug}_lower_bound"].values, df_test[f"{drug}_upper_bound"].values

    print(f"Minimizing {loss_type} loss")
    
    reg_param_lst = np.logspace(-6, 6, 13)
    losses_df = pd.DataFrame(columns=["alpha", "val_loss"])
    
    for i, alpha in enumerate(reg_param_lst):

        # fit a model on the training data using the given regularization parameter
        cv_model = CustomRidge(alpha=alpha)
        cv_model.fit(X_train, y_train, loss_type=loss_type, lower_bounds=lower_bounds_train, upper_bounds=upper_bounds_train)

        # get predictions on the test set, then compute binned mean squared error
        y_hat_cv = cv_model.predict(X_test)
        losses_df.loc[i, :] = [alpha, boundedLoss_Reg(y_hat_cv, y_test, lower_bounds_test, upper_bounds_test, loss_type=loss_type)]

    select_alpha = losses_df.sort_values("val_loss", ascending=True)["alpha"].values[0]
    
    print(f"Regularization parameter: {select_alpha}, minimum validation loss: {losses_df.sort_values('val_loss', ascending=True)['val_loss'].values[0]}")

    # fit a new model with the selected alpha parameter
    model = CustomRidge(alpha=select_alpha)
    model.fit(X_train, y_train, loss_type=loss_type, lower_bounds=lower_bounds_train, upper_bounds=upper_bounds_train)
    pickle.dump(model, open(os.path.join(ridge_dir, "model.sav"), "wb"))

    model = pickle.load(open(os.path.join(ridge_dir, "model.sav"), "rb"))
    y_pred = model.predict(X_test)
    
    summary_df = create_summary_df(df_test, y_pred, drug, binary_thresh, num_loci, model_name="LinReg", binarize=True, save_fName=os.path.join(ridge_dir, "test_predictions.csv"))
    summary_df["CV"] = 0
    
    bootstrap_df = []

    print("Performing bootstrapping...")
    for i in range(num_bootstrap):
        
        train_idx = np.random.choice(np.arange(0, len(X_train)), size=len(X_train), replace=True)
        
        X_bs = X_train[train_idx, :]
        y_bs = y_train[train_idx]
        lower_bounds_bs = lower_bounds_train[train_idx]
        upper_bounds_bs = upper_bounds_train[train_idx]
        
        # use regularization parameter determined above
        bs_model = CustomRidge(alpha=select_alpha)
        bs_model.fit(X_bs, y_bs, loss_type=loss_type, lower_bounds=lower_bounds_bs, upper_bounds=upper_bounds_bs)
        
        pickle.dump(bs_model, open(os.path.join(bootstrap_dir, f"model_{i}.sav"), "wb"))
        bs_model = pickle.load(open(os.path.join(bootstrap_dir, f"model_{i}.sav"), "rb"))
        y_pred_bs = bs_model.predict(X_test)
        
        bs_summary_df = create_summary_df(df_test, y_pred_bs, drug, binary_thresh, num_loci, "LinReg", binarize=True, save_fName=None)
        bs_summary_df["CV"] = i + 1
        bootstrap_df.append(bs_summary_df)
        
    bootstrap_df = pd.concat(bootstrap_df)
    final_df = pd.concat([summary_df, bootstrap_df], axis=0)
    final_df[["Lineage", "Num_Loci"]] = [int(include_lineage), num_loci]

    return final_df

In [31]:
results = ridge_mic(X_train, X_test, df_phenos, drug, include_lineage, binary_thresh, num_loci, num_bootstrap=10)
results_lineage = ridge_mic(X_train, X_test, df_phenos, drug, 1, binary_thresh, num_loci, num_bootstrap=10)

Minimizing L2 loss
Regularization parameter: 10.0, minimum validation loss: 2.496415928422395
Performing bootstrapping...
Fitting model with lineages
Minimizing L2 loss
Regularization parameter: 10.0, minimum validation loss: 2.493142405368175
Performing bootstrapping...


In [40]:
var = "Precision"
st.ttest_ind(results.query("CV > 0")[var], results_lineage.query("CV > 0")[var], equal_var=False)

Ttest_indResult(statistic=-1.3415765200582908, pvalue=0.19782880050582236)

In [2]:
# for drug in ["PZA", "EMB", "INH", "RIF", "LEV", "MXF"]:
#     if os.path.isdir(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/VCF_QC_files"):
#         shutil.rmtree(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/VCF_QC_files")

In [5]:
gene = 'clpC1'
h37Rv_genes.query("Symbol==@gene")#[["Start", "End"]].values[0]

,Gene_Ind,Feature,Start,End,Strand,Frame,H37rv_GeneID,Symbol,Function,Product,...,SWISS-MODEL,Orthologues M. leprae,Orthologues M. marinum,Orthologues M. smegmatis,Orthologues M. bovis,Orthologues M. lepromatosis,Orthologues M. tuberculosis,Orthologues M. abscessus,Orthologues M. haemophilum,Orthologues M. orygis
3696,3696,CDS,4038158,4040704,-,0.0,Rv3596c,clpC1,Hydrolyses proteins in presence of ATP. May in...,Probable ATP-dependent protease ATP-binding su...,...,P9WPC9,ML0235,MMAR_5100,MSMEG_6091,Mb3627c,NaN,NaN,NaN,NaN,NaN


In [5]:
who_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/WHO_catalog_clean.csv")


In [8]:
def get_qc_result(drug, gene):

    df = pd.read_csv(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/VCF_QC_files/{gene}_training_PASS_prop.txt", sep="\t", header=None)
    df = df[0].str.split(" ", expand=True)
    df[1] = df[1].replace("", np.nan)
    df[1] = df[1].astype(float)
    print(df.shape)
    return df

In [10]:
rpoA = get_qc_result("RIF", "rpoA")
rpoB = get_qc_result("RIF", "rpoB")
rpoC = get_qc_result("RIF", "rpoC")

(9135, 2)
(9135, 2)
(9135, 2)


In [12]:
rpoA.loc[~pd.isnull(rpoA[1])]

,0,1
49,ERR4810467,1.0
63,ERR4810928,1.0
77,ERR4810943,1.0
120,ERR4810993,1.0
178,ERR4810824,1.0
...,...,...
9007,SAMEA3558142,1.0
9015,SAMEA3558221,1.0
9053,SAMN02360616,1.0
9072,SAMN02381021,1.0


In [63]:
gene = 'rpoA'
h37Rv_genes.query("Symbol==@gene")[["Start", "End"]].values[0]

array([3877464, 3878507])

In [58]:
rpoBC_var.loc[(~pd.isnull(rpoBC_var[1])) & (rpoBC_var[1] < 0.75)]

,0,1
291,ERR4810502,0.0000
510,ERR4810731,0.0000
835,ERR4830638,0.6666
938,ERR4830933,0.6666
989,ERR4830991,0.6666
1006,ERR4831010,0.6666
1008,ERR4831011,0.6666
1031,ERR4830914,0.6666
1127,ERR4822438,0.7142
1169,ERR4831249,0.6666


In [ ]:
bcftools filter -i "POS >= 3877432 & POS <= 3878658" SAMEA3558209.vcf

In [7]:
who_variants.query("drug=='RIF' & confidence == '1) Assoc w R'")

,drug,genome_index,confidence,mutation
42,RIF,"761139,761140",1) Assoc w R,rpoB_p.His445Cys
43,RIF,761100,1) Assoc w R,rpoB_p.Gln432Lys
44,RIF,761101,1) Assoc w R,rpoB_p.Gln432Pro
45,RIF,761128,1) Assoc w R,rpoB_p.Ser441Leu
46,RIF,761101,1) Assoc w R,rpoB_p.Gln432Leu
47,RIF,"761139,761140",1) Assoc w R,rpoB_p.His445Ser
48,RIF,"761127,761128",1) Assoc w R,rpoB_p.Ser441Gln
49,RIF,"761109,761110",1) Assoc w R,rpoB_p.Asp435Phe
50,RIF,761102,1) Assoc w R,rpoB_p.Phe433dup
58,RIF,761134,1) Assoc w R,rpoB_p.Thr444dup


In [2]:
vcf_dir = "/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF"
isolates = os.listdir(vcf_dir)
len(isolates)

11115

In [37]:
train_isolates = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/train_isolate_variants.tsv", sep="\t")
len(train_isolates.Isolate.unique())

/tmp/ipykernel_32111/1310703126.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  train_isolates = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/train_isolate_variants.tsv", sep="\t")


11115

In [21]:
train_isolates.query("Isolate=='ERR4808977' & GENE in ['eis', 'rrs', 'rrl']")

,POS,REF,ALT,FILTER,AF,GENE,EFFECT,NUC,PROT,Isolate


In [17]:
cryptic = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/cryptic_data_curated_filtered.csv")
cc_df = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/criticalConcentrations_updated.csv")

In [15]:
cryptic.query("ROLLINGDB_ID=='ERR4808977'").columns

Index(['ROLLINGDB_ID', 'BIOSAMPLE_ACCESSION', 'ID', 'ISOLATION_LOCATION',
       'DB_OF_ORIGIN', 'STUDY_NAME', 'STUDY_PMID', 'TESTING_LOCATION', 'MEDIA',
       'AMI', 'CLO', 'EMB', 'ETA', 'INH', 'KAN', 'LEVO', 'LIN', 'MOXI', 'RIF',
       'RFB', 'DLM', 'BDQ', 'AMI_lower_bound', 'AMI_midpoint',
       'AMI_upper_bound', 'CLO_lower_bound', 'CLO_midpoint', 'CLO_upper_bound',
       'EMB_lower_bound', 'EMB_midpoint', 'EMB_upper_bound', 'ETA_lower_bound',
       'ETA_midpoint', 'ETA_upper_bound', 'INH_lower_bound', 'INH_midpoint',
       'INH_upper_bound', 'KAN_lower_bound', 'KAN_midpoint', 'KAN_upper_bound',
       'LEVO_lower_bound', 'LEVO_midpoint', 'LEVO_upper_bound',
       'LIN_lower_bound', 'LIN_midpoint', 'LIN_upper_bound',
       'MOXI_lower_bound', 'MOXI_midpoint', 'MOXI_upper_bound',
       'RIF_lower_bound', 'RIF_midpoint', 'RIF_upper_bound', 'RFB_lower_bound',
       'RFB_midpoint', 'RFB_upper_bound', 'DLM_lower_bound', 'DLM_midpoint',
       'DLM_upper_bound', 'BDQ_lower_boun

In [22]:
cryptic.query("ROLLINGDB_ID=='ERR4808977'")[['AMI', 'CLO', 'EMB', 'ETA', 'INH', 'KAN', 'LEVO', 'LIN', 'MOXI', 'RIF',
       'RFB', 'DLM', 'BDQ', 'KAN_quality']]

,AMI,CLO,EMB,ETA,INH,KAN,LEVO,LIN,MOXI,RIF,RFB,DLM,BDQ,KAN_quality
10197,2.0,<=0.24,1.25,NaN,<=0.05,5.0,0.5,0.5,0.06,<=0.06,<=0.25,<=0.002,<=0.02,HIGH


In [24]:
train_isolates.query("Isolate=='ERR4808977' & ~GENE.str.contains('PE')")

,POS,REF,ALT,FILTER,AF,GENE,EFFECT,NUC,PROT,Isolate
5766031,13626,A,ACCGCCCCGGTGAGTCCGGAGACTCTCTGATCTGAGACCTCAGCCG...,PASS,.,Rv0010c-Rv0011c,intergenic_region,n.13626_13627insCCGCCCCGGTGAGTCCGGAGACTCTCTGAT...,.,ERR4808977
5766032,14785,T,C,PASS,1.0,Rv0012,missense_variant,c.697T>C,p.Cys233Arg,ERR4808977
5766033,42417,T,C,PASS,1.0,Rv0039c-mtc28,intergenic_region,n.42417T>C,.,ERR4808977
5766034,55553,C,T,PASS,0.99,ponA1,missense_variant,c.1891C>T,p.Pro631Ser,ERR4808977
5766035,69989,G,A,PASS,1.0,Rv0064,missense_variant,c.1370G>A,p.Gly457Asp,ERR4808977
...,...,...,...,...,...,...,...,...,...,...
5766192,4095001,CG,C,PASS,0.98,Rv3655c,frameshift_variant,c.299delC,p.Ser100fs,ERR4808977
5766193,4095002,G,A,Del,1.0,Rv3655c,missense_variant,c.299C>T,p.Ser100Leu,ERR4808977
5766194,4100975,T,C,PASS,1.0,MTS2823-Rv3662c,intergenic_region,n.4100975T>C,.,ERR4808977
5766195,4359099,G,C,PASS,1.0,espK,missense_variant,c.684C>G,p.Ile228Met,ERR4808977


In [11]:
# for i, sample in enumerate(samples_no_VCF):

#     # get line counts of the FASTQ files
#     FQ1 = os.path.join(fastq_dir, sample, f"{sample}_R1.fastq.gz")
#     FQ2 = os.path.join(fastq_dir, sample, f"{sample}_R2.fastq.gz")

#     proc = subprocess.Popen(f"gunzip -c {FQ1} | wc -l", shell=True, encoding='utf8', stdout=subprocess.PIPE)
#     FQ1_line_count = int(proc.communicate()[0].rstrip("\n"))

#     proc = subprocess.Popen(f"gunzip -c {FQ2} | wc -l", shell=True, encoding='utf8', stdout=subprocess.PIPE)
#     FQ2_line_count = int(proc.communicate()[0].rstrip("\n"))

#     assert FQ1_line_count == FQ2_line_count
#     assert FQ2_line_count > 0

#     if i % 15 == 0:
#         print(i)

In [12]:
megapipe_samples = pd.read_csv("megapipe_samples.tsv", sep="\t", header=None)
assert len(samples_no_VCF) == len(megapipe_samples.loc[megapipe_samples[0].isin(samples_no_VCF)])

In [13]:
megapipe_samples.loc[megapipe_samples[0].isin(samples_no_VCF)].to_csv("megapipe_rerun.tsv", sep="\t", index=False, header=None)

In [15]:
megapipe_samples.loc[megapipe_samples[0]==samples_no_VCF[0]].to_csv("megapipe_test.tsv", sep="\t", index=False, header=None)

In [3]:
df_pza = pd.read_csv(os.path.join(data_dir, "PZA", "data_for_model.csv"))

In [4]:
df_pza.Lineage.value_counts()

Lineage
4          1061
2           108
3             9
1             2
BOVAFRI       2
Name: count, dtype: int64

In [19]:
mic_ml = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/MIC/MIC_ML_consortium_MIC_table.csv")
isolate_metadata = pd.read_csv("/n/data1/hms/dbmi/farhat/rollingDB/metadata/isolate_metadata.csv")

In [11]:
found = []

for i, row in mic_ml.iterrows():

    isolate = row['ROLLINGDB_ID']

    if pd.isnull(isolate):
        isolate = row['ID']

    if pd.isnull(isolate):
        isolate = row['BIOSAMPLE_ACCESSION']

    if pd.isnull(isolate):
        print(f"{isolate} not found!")
    
    # 
    if os.path.isdir(f"/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/{isolate}"):

        if os.path.isfile(f"/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/{isolate}/{isolate}_R1.fastq.gz") and os.path.isfile(f"/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/{isolate}/{isolate}_R2.fastq.gz"):
            
            found.append(isolate)
        else:
            print(isolate)

len(found)/len(mic_ml)

20802
23155
31309


0.8028933092224232

In [8]:
for drug in ["RIF", "INH", "EMB", "LEV", "MXF"]:

    print(f"python3 data_processing/07_clean_training_data.py {drug}")
    print(f"python3 data_processing/08_combine_datasets.py config_files/config_{drug.lower()}.yaml")

python3 data_processing/07_clean_training_data.py RIF
python3 data_processing/08_combine_datasets.py config_files/config_rif.yaml
python3 data_processing/07_clean_training_data.py INH
python3 data_processing/08_combine_datasets.py config_files/config_inh.yaml
python3 data_processing/07_clean_training_data.py EMB
python3 data_processing/08_combine_datasets.py config_files/config_emb.yaml
python3 data_processing/07_clean_training_data.py LEV
python3 data_processing/08_combine_datasets.py config_files/config_lev.yaml
python3 data_processing/07_clean_training_data.py MXF
python3 data_processing/08_combine_datasets.py config_files/config_mxf.yaml


In [2]:
megapipe_samples = pd.read_csv("megapipe_samples.tsv", sep="\t", header=None)

In [71]:
train_isolates = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/train_isolate_variants.tsv", sep="\t")
len(train_isolates.Isolate.unique())

/tmp/ipykernel_30781/1310703126.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  train_isolates = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/train_isolate_variants.tsv", sep="\t")


11042

In [44]:
# for drug in ["PZA", "RIF", 'INH', 'EMB', 'LEV', 'MXF']:

    # if os.path.isdir(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/fastas"):
    #     shutil.rmtree(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/fastas")
    #     print(drug)

    # if os.path.isdir(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/VCF_QC_files"):
    #     shutil.rmtree(f"/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/{drug}/VCF_QC_files")
    #     print(drug)

In [54]:
11115-11041

74

In [34]:
vcf_dir="/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/VCF"

isolates = os.listdir(vcf_dir)
# fNames = list(set(fNames) - set(train_isolates.Isolate))
len(isolates)

11115

In [34]:
missing_files = []

for i, isolate in enumerate(isolates):

    assert os.path.isfile(os.path.join(vcf_dir, isolate, f"pilon/{isolate}.vcf"))
    assert os.path.isfile(os.path.join(vcf_dir, isolate, f"pilon/{isolate}.eff.vcf"))

    line_count = subprocess.Popen(f'grep -v "^#" {vcf_dir}/{isolate}/pilon/{isolate}.vcf | wc -l', shell=True, stdout=subprocess.PIPE, encoding='utf8')
    line_count = int(line_count.communicate()[0].replace('\n', ''))

    if line_count == 0:
        missing_files.append(isolate)

    if i % 1000 == 0:
        print(i)
        
print(len(missing_files))

0
1000
2000
3000
4000
5000
6000
7000
8000
9000
10000
11000
0


In [20]:
assert len(megapipe_samples.loc[megapipe_samples[0].isin(missing_files)]) == len(missing_files)

In [22]:
megapipe_samples.loc[megapipe_samples[0].isin(missing_files)].to_csv("megapipe_rerun.tsv", sep="\t", index=False, header=None)

In [ ]:
# for isolate in missing_files:
#     shutil.rmtree(f"{vcf_dir}/{isolate}")

In [26]:
missing_files[0][:6]

'ERR483'

In [37]:
# for isolate in [missing_files[0]]:

#     print(isolate)
#     out_dir = f"/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/{isolate}"
#     assert os.path.isdir(out_dir)

#     # os.remove(os.path.join(out_dir, f"{isolate}_R1.fastq.gz"))
#     # os.remove(os.path.join(out_dir, f"{isolate}_R2.fastq.gz"))

#     if not os.path.isfile(f"{out_dir}/{isolate}_R1.fastq.gz"):
#         subprocess.run(f"wget ftp://ftp.sra.ebi.ac.uk/vol1/fastq/{isolate}/{isolate}/{isolate}_1.fastq.gz -c -O {out_dir}/{isolate}_R1.fastq.gz", shell=True)
#         subprocess.run(f"wget ftp://ftp.sra.ebi.ac.uk/vol1/fastq/{isolate}/{isolate}/{isolate}_2.fastq.gz -c -O {out_dir}/{isolate}_R2.fastq.gz", shell=True)

In [47]:
# for isolate in missing_files:

#     if os.path.isfile(f"/n/data1/hms/dbmi/farhat/rollingDB/cryptic_output/{isolate}/pilon/{isolate}.vcf"):
#         fName = f"/n/data1/hms/dbmi/farhat/rollingDB/cryptic_output/{isolate}/pilon/{isolate}.vcf"
    
#     elif os.path.isfile(f"/n/data1/hms/dbmi/farhat/rollingDB/genomic_data/{isolate}/pilon/{isolate}.vcf"):
#         fName = f"/n/data1/hms/dbmi/farhat/rollingDB/genomic_data/{isolate}/pilon/{isolate}.vcf"

#     else:
#         print(f"No VCF for {isolate}")

#     # check that more than 75% reads aligned to MTBG
#     kraken_report = pd.read_csv(os.path.join(vcf_dir, isolate, "kraken/kraken_report"), sep="\t", header=None)
#     assert len(kraken_report.loc[kraken_report[5].str.contains('Mycobacterium tuberculosis complex')]) > 0
    
#     # this is classified read proportion -- ranges from 0 to 100
#     assert kraken_report.loc[kraken_report[5].str.contains('Mycobacterium tuberculosis complex')][0].values[0] >= 90

#     # check that original VCF file is not empty
#     line_count = subprocess.Popen(f'grep -v "^#" {fName} | wc -l', shell=True, stdout=subprocess.PIPE, encoding='utf8')
#     line_count = int(line_count.communicate()[0].replace('\n', ''))

#     if line_count == 0:
#         print(f"VCF for {isolate} is empty")

In [59]:
missing_files[0]

'ERR4831159'

In [58]:
missing_files[-1]

'ERR4829685'

In [57]:
'ERR4797007' in missing_files

True

In [69]:
pd.DataFrame({"Isolate": "ERR4797007",
              "FQ1": '/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ERR4797007/ERR4797007_R1.fastq',
              "FQ2": '/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ERR4797007/ERR4797007_R2.fastq'
             }, index=[0]).to_csv("/home/sak0914/MtbQuantCNN/megapipe_test.tsv", sep="\t", index=False, header=None)

In [3]:
megapipe_samples.loc[megapipe_samples[0]=="ERR4797007"].to_csv("/home/sak0914/MtbQuantCNN/megapipe_test.tsv", sep="\t", index=False, header=None)

In [6]:
megapipe_samples.loc[megapipe_samples[0]=="ERR4797007"][1].values

array(['/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ERR4797007/ERR4797007_R1.fastq.gz'],
      dtype=object)

In [7]:
os.path.isfile('/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ERR4797007/ERR4797007_R1.fastq.gz')

True

In [23]:
with open(os.path.join(vcf_dir, isolate, "kraken/kraken_report"), "r") as file:
    lines = file.readlines()

In [45]:
kraken_report.loc[kraken_report[5].str.contains('Mycobacterium tuberculosis')][5].values

array(['                  Mycobacterium tuberculosis complex',
       '                    Mycobacterium tuberculosis',
       '                      Mycobacterium tuberculosis F11',
       '                      Mycobacterium tuberculosis CDC1551',
       '                      Mycobacterium tuberculosis H37Rv',
       '                      Mycobacterium tuberculosis CTRI-2'],
      dtype=object)

99.81

In [31]:
kraken_report.loc[kraken_report[5].str.contains('&'.join(['Mycobacterium', 'tuberculosis']), case=False)]

,0,1,2,3,4,5


In [12]:
missing_files[0]

'ERR4831159'

In [16]:
megapipe_samples.loc[megapipe_samples[0].isin(missing_files)]

,0,1,2
384,ERR4831159,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
388,ERR4797734,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
409,ERR4830142,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
410,ERR4797025,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
540,ERR4811936,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
...,...,...,...
9831,ERR4822571,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
9920,ERR3287805,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
10079,ERR4812332,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
10358,ERR4828990,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...,/n/data1/hms/dbmi/farhat/rollingDB/fastq_db/ER...
